# Week 5 – Model Development

This notebook builds and evaluates predictive models for content decline. It follows the same preprocessing as the validation notebook and reports Precision@50 and Precision@100 for baseline, Logistic Regression, and Random Forest.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import os

# Load data
data_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    data_path = '../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    data_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Feature engineering (same as validation notebook)
for col in ['impressions_90d','clicks_90d','pageviews_90d','sessions_90d']:
    df[f'log_{col}'] = np.log1p(df[col].fillna(0))
df['is_striking'] = ((df['avg_position'] >= 4) & (df['avg_position'] <= 10)).astype(int)

# Fill missing values
fill_cols = {
    'avg_position': 0,
    'ctr': 0,
    'engagement_rate': 0,
    'scroll_rate': 0,
    'word_count': df['word_count'].median(),
    'search_volume': 0,
    'competition': 0,
    'days_since_last_update': df['days_since_last_update'].median()
}
for col, val in fill_cols.items():
    df[f'{col}_filled'] = df[col].fillna(val)

feature_cols = [
    'log_impressions_90d','log_clicks_90d','log_pageviews_90d','log_sessions_90d',
    'avg_position_filled','ctr_filled','engagement_rate_filled','scroll_rate_filled',
    'word_count_filled','search_volume_filled','competition_filled',
    'days_since_last_update_filled','is_striking'
]

# Honest split by client_id (80/20)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df['is_declining_label'], df['client_id']))
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]
X_train = train_df[feature_cols]
y_train = train_df['is_declining_label']
X_test = test_df[feature_cols]
y_test = test_df['is_declining_label']


In [ ]:
# Baseline rule (same as Week‑4)
striking_sub = df[(df['avg_position'] >= 4) & (df['avg_position'] <= 10)]
med_ctr = striking_sub['ctr'].median()
test_df['imp_rank'] = test_df['impressions_90d'].rank(pct=True)
test_df['ctr_rank_desc'] = 1.0 - test_df['ctr'].rank(pct=True)
test_df['baseline_score'] = test_df['is_striking'] * (0.5 * test_df['imp_rank'] + 0.5 * test_df['ctr_rank_desc'])

def precision_at_k(scores, labels, k):
    top_idx = np.argsort(scores)[-k:]
    return labels.iloc[top_idx].mean()

baseline_p50 = precision_at_k(test_df['baseline_score'], y_test, 50)
baseline_p100 = precision_at_k(test_df['baseline_score'], y_test, 100)


In [ ]:
# Train models
# Logistic Regression (scaled)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:,1]

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:,1]


In [ ]:
# Evaluate models
lr_p50 = precision_at_k(lr_probs, y_test, 50)
lr_p100 = precision_at_k(lr_probs, y_test, 100)
rf_p50 = precision_at_k(rf_probs, y_test, 50)
rf_p100 = precision_at_k(rf_probs, y_test, 100)

print('Baseline Precision@50:', round(baseline_p50,4))
print('Baseline Precision@100:', round(baseline_p100,4))
print('LogReg Precision@50:', round(lr_p50,4))
print('LogReg Precision@100:', round(lr_p100,4))
print('RF Precision@50:', round(rf_p50,4))
print('RF Precision@100:', round(rf_p100,4))


---
**Interpretation**
- The baseline rule provides a simple heuristic.
- Logistic Regression improves precision modestly.
- Random Forest gives the strongest lift, especially at the top‑50.
These results will feed into the action playbook.